In [1]:
import sys, torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn as nn
sys.path.append('../')
from utilities import convert_Pass2tensor, generate_matrix, load_embedding, print_exams
from torch.utils.data import Dataset, DataLoader
from DMS_utilities import fasta_to_dict

In [2]:
def load_crick(path, suffix, id_cols, keep_cols, rename_map, subset=False):
    df = pd.read_csv(path)

    # 只对存在的 id 列加后缀（以防有的文件列不全）
    existing_id_cols = [c for c in id_cols if c in df.columns]
    df[existing_id_cols] = df[existing_id_cols].add(f'_{suffix}')

    if subset:
        # 只选文件里真正存在的列，避免 KeyError
        existing_keep_cols = [c for c in keep_cols if c in df.columns]
        df = df[existing_keep_cols]

        # 只重命名实际存在的列
        effective_rename = {k: v for k, v in rename_map.items() if k in df.columns}
        if effective_rename:
            df = df.rename(columns=effective_rename)

    return df

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]


In [3]:
import torch
import pandas as pd

device = torch.device('cuda:0')
# id_cols = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d']
# keep_cols = id_cols + ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName',
#                         'serumDate', 'virusDate','serumType', 'serumIslID', 'virusIslID', 'HI_Dist']
# rename_map = {'serumType': 'Type', 'HI_Dist': 'label',}

# Crick_early = load_crick('../../data/data_40/all.csv', 'early', id_cols, keep_cols, rename_map)
# Crick_2024NH = load_crick('../../data/data_40/Vaccination_41.csv', '2024NH', id_cols, keep_cols, rename_map)
# Crick_2024SH = load_crick('../../data/data_40/Vaccination_42.csv', '2024SH', id_cols, keep_cols, rename_map)
# Crick_2025NH = load_crick('../../data/data_40/Vaccination_43.csv', '2025NH', id_cols, keep_cols, rename_map)
dict_path = '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/src/Section5_DMS/zhulin/'
# dataset_name = 'H3N2_vac2025_HA_17-345_dms'
dataset_name = 'H3N2_HA_epi202510-202602_origin'
sequences = fasta_to_dict(dict_path + 'sequence/' + dataset_name + '.fasta')

In [11]:
DMS_sequences = fasta_to_dict(dict_path + 'sequence/H3N2_vac2025_HA_17-345_dms.fasta')
emb_dict_vac = load_embedding(dict_path + 'embedding/H3N2_vac2025_HA_17-345_dms')

Loading tensor: 100%|██████████| 6253/6253 [13:26<00:00,  7.75file/s]


In [4]:
emb_dict_DMS = load_embedding(dict_path + 'embedding/' + dataset_name)

Loading tensor: 100%|██████████| 11122/11122 [24:22<00:00,  7.61file/s]


In [5]:
class Emb2Vec(nn.Module):
    def __init__(self, model):
        super(Emb2Vec, self).__init__()
        self.matrix_dropout = model.matrix_dropout
        self.matrix_pooler = model.matrix_pooler
        self.linear = model.linear[0][0]
        self.Gelu = model.linear[0][1]
    def forward(self, embedding):
        embedding = self.matrix_dropout(embedding)
        seq_vector = self.matrix_pooler(embedding)
        seq_vector = self.linear(seq_vector)
        return seq_vector
    
def run_emb2vec_on_dict(emb_dict, emb2vec, device='cpu'):
    emb2vec = emb2vec.to(device)
    emb2vec.eval()

    vec_dict = {}
    for k, emb in emb_dict.items():
        emb = emb.to(device)
        if emb.dim() == 2:   # (566, 2560) -> (1, 566, 2560)
            emb = emb.unsqueeze(0)

        with torch.no_grad():
            vec = emb2vec(emb)[0].cpu()   # (D_out,)
        vec_dict[k] = vec
    return vec_dict


In [6]:
device = torch.device("cuda:0")
model = torch.load('../../trained_model/1.7_Artificial_back/2025-08-19_17-43-32.pth', weights_only=False, map_location=device)

In [12]:
embedding_pooler = Emb2Vec(model)
vec_dict = run_emb2vec_on_dict(emb_dict_DMS, embedding_pooler, device=device)
vec_dict_vac = run_emb2vec_on_dict(emb_dict_vac, embedding_pooler, device=device)

In [13]:
vec_dict_final = vec_dict | vec_dict_vac

In [ ]:
vec_dict.pop('matrix_EPI_ISL_19296516')

tensor([ 0.3504, -0.1991, -1.9472, -1.9416, -0.4024,  0.1602, -0.0895,  0.1836,
        -1.8830, -0.0976, -0.2940,  0.0459,  0.6208, -0.7497, -2.1838, -1.3190,
         0.1364, -0.0198, -0.4357, -1.6481, -1.8998,  0.2251, -0.2939,  0.0393,
         0.1083,  0.0269, -2.2381, -0.0129, -0.3338,  0.0096, -0.0561, -0.1083,
         0.2466,  0.0724,  0.0595, -1.3005,  0.1821, -0.4776, -0.1547, -1.4315,
        -0.5655, -0.0461, -0.2754,  0.1101, -1.4509,  0.1791, -0.0723,  0.0626,
        -0.0953,  0.0101, -2.0151, -0.5738, -0.0775,  0.2187,  0.3596, -2.6782,
        -0.1295,  0.3854,  0.2472, -0.0155,  0.3260, -0.1261, -1.7902, -0.3005,
        -0.7105, -0.0389, -0.5535, -1.1421, -0.0419,  0.4171,  0.1077, -0.3743,
        -1.1411,  0.0430, -0.0567, -0.2362, -0.0390,  0.1686, -1.7373, -0.2853,
        -0.5884, -0.8061, -0.1644, -0.7308,  0.0094, -0.7566, -0.7746, -0.1660,
        -2.1352,  0.2889,  0.2129, -0.0092, -0.4207, -0.2987,  0.3972, -0.8244,
        -0.8623,  0.1009,  0.2597, -1.60

In [14]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

keys = list(vec_dict_final.keys())
X = np.stack([vec_dict_final[k] for k in keys])  # (N, 256)

pca_30 = PCA(n_components=30, random_state=42)
X_30 = pca_30.fit_transform(X)
print("Explained variance (30 PCs):", float(pca_30.explained_variance_ratio_.sum()))

pca_2 = PCA(n_components=2, random_state=42)
X_2d = pca_2.fit_transform(X_30)

target_key = "matrix_origin"
is_target = np.array([k == target_key for k in keys])

fig = go.Figure()

# 先加“其它点”trace
fig.add_trace(go.Scatter(
    x=X_2d[~is_target, 0],
    y=X_2d[~is_target, 1],
    mode="markers",
    name="others",
    text=np.array(keys)[~is_target],     # hover 显示 key
    hovertemplate="%{text}<extra></extra>",
    marker=dict(size=6, opacity=0.8),
))

# 再加“matrix_HA”trace（红色，最后加=显示在最上层）
if is_target.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_target, 0],
        y=X_2d[is_target, 1],
        mode="markers",
        name=target_key,
        text=np.array(keys)[is_target],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, color="red", opacity=1.0, line=dict(width=1, color="black")),
    ))
else:
    print(f"Warning: key '{target_key}' not found in vec_dict.")

fig.update_layout(
    title="2D embedding (PCA) — hover to show key",
    xaxis_title="Dim 1",
    yaxis_title="Dim 2",
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

Explained variance (30 PCs): 0.9772464632987976


In [13]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import plotly.graph_objects as go

keys = list(vec_dict.keys())
X = np.stack([vec_dict[k] for k in keys])  # (N, 256)

pca_30 = PCA(n_components=30, random_state=42)
X_30 = pca_30.fit_transform(X)
print("Explained variance (30 PCs):", float(pca_30.explained_variance_ratio_.sum()))

# -----------------------------
# 1) 在 30D PCA 空间里找离群点（鲁棒 Mahalanobis）
# -----------------------------
mcd = MinCovDet(random_state=42).fit(X_30)
md2 = mcd.mahalanobis(X_30)  # squared Mahalanobis distance，越大越离群

# 阈值：优先用卡方阈值（更“统计学”），没有 scipy 就用分位数兜底
try:
    from scipy.stats import chi2
    thr = chi2.ppf(0.99, df=X_30.shape[1])  # 99% 置信阈值
except Exception:
    thr = np.quantile(md2, 0.99)

is_outlier = md2 > thr
outlier_keys = np.array(keys)[is_outlier]

print(f"Outliers (n={is_outlier.sum()}), threshold={thr:.3f}")
for k, s in sorted(zip(outlier_keys, md2[is_outlier]), key=lambda x: x[1], reverse=True):
    print(f"{k}\tmd2={s:.3f}")

# 如果你只想看“最离群的前 K 个”
K = 20
topk_idx = np.argsort(-md2)[:K]
print(f"\nTop-{K} most outlying points:")
for i in topk_idx:
    print(f"{keys[i]}\tmd2={md2[i]:.3f}")

# -----------------------------
# 2) 继续做 2D 展示（和你原来一致）
# -----------------------------
pca_2 = PCA(n_components=2, random_state=42)
X_2d = pca_2.fit_transform(X_30)

target_key = "matrix_HA"
is_target = np.array([k == target_key for k in keys])

fig = go.Figure()

# others（排除 target 和 outliers，避免重复显示）
mask_others = (~is_target) & (~is_outlier)
fig.add_trace(go.Scatter(
    x=X_2d[mask_others, 0],
    y=X_2d[mask_others, 1],
    mode="markers",
    name="others",
    text=np.array(keys)[mask_others],
    hovertemplate="%{text}<extra></extra>",
    marker=dict(size=6, opacity=0.8),
))

# outliers（单独一层）
if is_outlier.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_outlier, 0],
        y=X_2d[is_outlier, 1],
        mode="markers",
        name="outliers",
        text=np.array(keys)[is_outlier],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, symbol="x", opacity=1.0, line=dict(width=1)),
    ))

# target（最上层）
if is_target.any():
    fig.add_trace(go.Scatter(
        x=X_2d[is_target, 0],
        y=X_2d[is_target, 1],
        mode="markers",
        name=target_key,
        text=np.array(keys)[is_target],
        hovertemplate="%{text}<extra></extra>",
        marker=dict(size=10, color="red", opacity=1.0, line=dict(width=1, color="black")),
    ))
else:
    print(f"Warning: key '{target_key}' not found in vec_dict.")

fig.update_layout(
    title="2D embedding (PCA) — hover to show key (outliers marked as x)",
    xaxis_title="Dim 1",
    yaxis_title="Dim 2",
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()


Explained variance (30 PCs): 0.8042601943016052
Outliers (n=2945), threshold=50.892
matrix_S235F	md2=11626.469
matrix_S235W	md2=10182.162
matrix_T26M	md2=8175.361
matrix_S235H	md2=7813.745
matrix_S235C	md2=7246.877
matrix_S235Y	md2=7198.392
matrix_V325I	md2=6664.035
matrix_S235R	md2=6538.424
matrix_S235K	md2=6343.311
matrix_S235M	md2=6262.622
matrix_S235D	md2=6158.270
matrix_S235L	md2=5906.518
matrix_S235P	md2=5898.804
matrix_K156I	md2=5882.297
matrix_S235I	md2=5743.413
matrix_T26D	md2=5576.545
matrix_S235T	md2=5483.272
matrix_R224K	md2=5338.264
matrix_T26E	md2=5094.398
matrix_S235G	md2=4969.276
matrix_S235A	md2=4800.971
matrix_R224D	md2=4706.965
matrix_F211Y	md2=4474.636
matrix_T151K	md2=4440.300
matrix_S235V	md2=4379.676
matrix_A228T	md2=4360.563
matrix_F208I	md2=4348.312
matrix_T151A	md2=4233.493
matrix_S235Q	md2=4232.491
matrix_N161S	md2=4180.235
matrix_R224E	md2=4130.402
matrix_T151R	md2=4128.686
matrix_S235N	md2=4009.649
matrix_S235E	md2=3906.872
matrix_T26N	md2=3890.516
matrix_F

In [14]:
import re
from collections import defaultdict, Counter

def summarize_sites(names, prefix="matrix_", top_n=20, min_count=1, ignore_X=False):
    """
    names: 例如 outlier_keys 或 keys
    prefix: key 前缀（默认 matrix_）
    ignore_X: 是否忽略 'X' 这类未知/终止标记（按你数据含义决定）
    """
    # 兼容：matrix_S235F / matrix_R224X / matrix_E292N ...
    pat = re.compile(rf"^{re.escape(prefix)}([A-Z])(\d+)([A-Z\*X])$")

    site_ref = defaultdict(set)         # pos -> {refAA}
    site_alt_counter = defaultdict(Counter)  # pos -> Counter({altAA: count})
    bad = []

    for name in names:
        m = pat.match(name)
        if not m:
            bad.append(name)
            continue
        ref, pos, alt = m.group(1), int(m.group(2)), m.group(3)
        if ignore_X and alt == "X":
            continue
        site_ref[pos].add(ref)
        site_alt_counter[pos][alt] += 1

    # 汇总
    rows = []
    for pos, alt_ct in site_alt_counter.items():
        total = sum(alt_ct.values())
        if total < min_count:
            continue
        refs = "".join(sorted(site_ref[pos]))  # 理论上应只有1个ref；如果出现多个，说明命名/对齐可能混了
        rows.append((total, pos, refs, alt_ct))

    rows.sort(reverse=True, key=lambda x: x[0])

    print(f"Parsed mutations: {len(names)-len(bad)} / {len(names)} (unparsed={len(bad)})")
    if bad:
        print("Examples of unparsed keys:", bad[:10])

    print("\nHigh-frequency sites:")
    for total, pos, refs, alt_ct in rows[:top_n]:
        alt_str = ", ".join([f"{aa}:{cnt}" for aa, cnt in alt_ct.most_common()])
        print(f"pos {pos} (ref={refs})  n={total}  alts=[{alt_str}]")

    return rows

# ===== 用法 1：只统计离群点里高频位点 =====
# outlier_keys = np.array(keys)[is_outlier]  # 你之前已经有了就直接用
rows_outliers = summarize_sites(outlier_keys, top_n=30, min_count=5, ignore_X=False)

# ===== 用法 2：统计全体 keys 里高频位点 =====
# rows_all = summarize_sites(keys, top_n=30, min_count=10, ignore_X=False)


Parsed mutations: 2945 / 2945 (unparsed=0)

High-frequency sites:
pos 113 (ref=C)  n=19  alts=[K:1, Q:1, G:1, L:1, V:1, T:1, N:1, E:1, S:1, I:1, A:1, M:1, W:1, P:1, F:1, D:1, R:1, H:1, Y:1]
pos 68 (ref=C)  n=19  alts=[G:1, K:1, Q:1, L:1, V:1, T:1, N:1, S:1, I:1, E:1, M:1, W:1, A:1, F:1, P:1, R:1, H:1, D:1, Y:1]
pos 241 (ref=D)  n=19  alts=[K:1, Q:1, G:1, L:1, V:1, T:1, N:1, E:1, S:1, I:1, A:1, M:1, W:1, P:1, F:1, R:1, H:1, Y:1, C:1]
pos 208 (ref=F)  n=19  alts=[L:1, V:1, G:1, K:1, Q:1, S:1, I:1, E:1, T:1, N:1, P:1, M:1, W:1, A:1, Y:1, C:1, R:1, H:1, D:1]
pos 197 (ref=G)  n=19  alts=[W:1, M:1, A:1, F:1, P:1, H:1, R:1, D:1, C:1, Y:1, Q:1, K:1, V:1, L:1, N:1, T:1, I:1, S:1, E:1]
pos 216 (ref=G)  n=19  alts=[H:1, R:1, D:1, C:1, Y:1, W:1, M:1, A:1, F:1, P:1, N:1, T:1, I:1, S:1, E:1, Q:1, K:1, V:1, L:1]
pos 199 (ref=H)  n=19  alts=[A:1, M:1, W:1, P:1, F:1, D:1, R:1, Y:1, C:1, K:1, Q:1, G:1, L:1, V:1, T:1, N:1, E:1, S:1, I:1]
pos 176 (ref=I)  n=19  alts=[K:1, Q:1, G:1, L:1, V:1, T:1, N:1, E:1

In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# rows_outliers: summarize_sites(...) 的返回值
top_k = 25
rows = rows_outliers[:top_k]

# -------------------------
# 1) 高频位点：频次柱状图
# -------------------------
df_site = pd.DataFrame({
    "pos": [r[1] for r in rows],
    "ref": [r[2] for r in rows],
    "n":   [r[0] for r in rows],
})
df_site["label"] = df_site["pos"].astype(str) + " (" + df_site["ref"] + ")"
df_site = df_site.sort_values("n", ascending=False)

fig1 = px.bar(
    df_site,
    x="label", y="n",
    hover_data={"pos": True, "ref": True, "n": True},
    title=f"High-frequency mutation sites (Top {top_k})",
)
fig1.update_layout(
    xaxis_title="Site (ref)",
    yaxis_title="Count",
    xaxis_tickangle=-45,
)
fig1.show()

# -------------------------
# 2) 每个位点：有哪些突变（堆叠柱状图）
# -------------------------
records = []
for total, pos, ref, alt_ct in rows:
    for alt, cnt in alt_ct.items():
        records.append({"pos": pos, "ref": ref, "alt": alt, "count": cnt})

df_alt = pd.DataFrame(records)
df_alt["label"] = df_alt["pos"].astype(str) + " (" + df_alt["ref"] + ")"

# 保持 x 轴顺序与 df_site 一致
order = df_site["label"].tolist()
df_alt["label"] = pd.Categorical(df_alt["label"], categories=order, ordered=True)

# 让 X（未知/终止）尽量排到最后（可按需改）
alts = sorted(df_alt["alt"].unique(), key=lambda a: (a == "X", a))

fig2 = go.Figure()
for a in alts:
    sub = df_alt[df_alt["alt"] == a]
    fig2.add_trace(go.Bar(x=sub["label"], y=sub["count"], name=a))

fig2.update_layout(
    barmode="stack",
    title=f"Mutation spectrum per site (stacked, Top {top_k})",
    xaxis_title="Site (ref)",
    yaxis_title="Count",
    xaxis_tickangle=-45,
)
fig2.show()

# -------------------------
# 3) 位点 × 突变氨基酸：热图（更适合看“有哪些突变”）
# -------------------------
pivot = df_alt.pivot_table(index="alt", columns="label", values="count", aggfunc="sum", fill_value=0)

fig3 = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=pivot.index,
))
fig3.update_layout(
    title=f"Heatmap of mutations (alt × site, Top {top_k})",
    xaxis_title="Site (ref)",
    yaxis_title="Alt AA",
    xaxis_tickangle=-45,
)
fig3.show()


/tmp/ipykernel_900853/444019205.py:69: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

